# Fine-Tuning Genomic LLMs with LoRA

---

## Learning Outcomes

By the end of this session, participants will be able to:
- configure a LoRA adapter with the `peft` (parameter efficient fine-tuning) library — rank, scaling, target modules
- fine-tune a pretrained genomic transformer with the Hugging Face `Trainer` API while keeping ~99% of its weights frozen
- save a lightweight adapter, reload it on a fresh trunk, and merge it for inference
- evaluate and probe a LoRA-tuned sequence classifier

## credits

This notebook is adapted from [Meg Hedge's ECCB2026 tutorial](https://github.com/meghegde/genomic-llms-in-practice/tree/main). It is the parameter-efficient counterpart of notebook 03: same model, same task, same data, same metrics — only the fine-tuning regime changes.

In nb03 we updated **every** weight of DNABERT. Here we freeze the pretrained trunk entirely and train only small **Low-Rank Adapters** (LoRA)injected inside the attention layers: [LoRA](https://arxiv.org/abs/2106.09685) (Hu et al., 2021). Everything before and after training is reproduced unchanged, so the two notebooks form a controlled comparison: the same accuracy question at ~1% of the trainable parameters.


---

We again use [DNABERT (6-mer)](https://huggingface.co/zhihan1996/DNA_bert_6) on promoter classification, and this session again requires a GPU (a T4 on Colab or Kaggle is enough).

## Install and import required packages

In [ ]:
# Installing required libraries
!pip -q install transformers datasets evaluate accelerate scikit-learn seaborn matplotlib peft

In [ ]:
# Importing required libraries
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

from sklearn.metrics import accuracy_score, f1_score

## Why parameter-efficient fine-tuning?

In the previous sessions we used pretrained models **off-the-shelf**. Fine-tuning keeps everything the model learned during pre-training and adapts it to a specific task using a small labelled dataset — transfer learning.

![](../images/dallatorre2023_fig1.png)

Figure 1 c. & d. from [Dalla-Torre et al.](https://www.biorxiv.org/content/10.1101/2023.01.11.523679v2)

Notebook 02 corresponds to the top sub-figure: frozen LLM, external classifier. Nb03's full fine-tuning corresponds to the bottom: every weight moves. **LoRA sits in between**: the trunk stays frozen — it *cannot* be overwritten — but trainable paths are injected *inside* the network, so adaptation still happens in the feature space, not merely at the output.

**The idea** ([Hu et al., 2021](https://arxiv.org/abs/2106.09685)): for each selected pre-trained weight matrix `W`, keep `W` frozen and add a low-rank product of two small trainable matrices:

```
W' x  =  W x  +  (alpha / r) · B @ A @ x        A ∈ R^(r×d),  B ∈ R^(d×r),  r ≪ d
```

A and B are  **adapters** which are added on top of the existing model.

![](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/peft/lora_animated.gif)

If the adapters were the same dimension than the original matrices, we would not gain anything, instead the adapters are **low-rank**.

The idea of low-rank factorization is that a large matrix can be approximately represented by the multiplication of 2 smaller matrix (A and B)
![](https://upload.wikimedia.org/wikipedia/commons/0/0c/Low-rank_matrix_approximation_illustrating_LoRA.gif?utm_source=en.wikipedia.org&utm_campaign=index&utm_content=thumbnail_unscaled)


For example: a 768×768 attention projection has 590k parameters; its rank-16 adapter has 2 × 768 × 16 ≈ 25k : a 24x reduction *per targeted matrix*, while the trunk's ~90M parameters stay byte-identical to the checkpoint. Three properties make this practical:

- **`B` is initialized to zero**, so adapters start as an exact no-op: training begins from the precise pre-trained function, with no random-perturbation warm-up
- **the trunk's gradients never need to be stored** — a memory lever we will measure below, mirroring nb03's batch-size probe
- **the trained delta merges exactly into `W`** after training: zero inference overhead, and one trunk can serve many tasks by swapping adapters

## The Model: DNABERT (6-mer)

Same as before, we fine-tune [`zhihan1996/DNA_bert_6`](https://huggingface.co/zhihan1996/DNA_bert_6), the 6-mer checkpoint of **DNABERT** ([Ji et al., 2021](https://doi.org/10.1093/bioinformatics/btab083); [code](https://github.com/jerryji1993/DNABERT); [huggingface](huggingface.co/zhihan1996/DNA_bert_6)):



In [ ]:
MODEL_NAME = "zhihan1996/DNA_bert_6"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## The Task: Promoter Classification

Promoters are short regulatory regions at the start of genes: they tell the transcription machinery where to bind and from which nucleotide to begin reading the gene. Predicting them from sequence alone is a classic binary classification problem — given a window of DNA, is it a promoter or genomic background?

### Dataset

We use the `NT_promoter_all` configuration of the Genomic Benchmarks collection on Hugging Face: [datasets/darlednik/geneb-tasks](https://huggingface.co/datasets/darlednik/geneb-tasks)

- **300 bp human DNA windows**, centered on annotated transcription start sites for positives versus random genomic background for negatives
- **53,276 train / 5,920 test** sequences; label `1` = promoter, `0` = non-promoter
- from the Nucleotide Transformer benchmark suite ([Dalla-Torre et al., 2023](https://www.biorxiv.org/content/10.1101/2023.01.11.523679v2)) — the same paper as the figure above
- the repository aggregates ~100 benchmark configs (`NT_*`, `GB_*`, `GUE_*`, ...): splice sites, enhancers, histone marks, ... — a ready-made playground if you want to fine-tune on a second task

In [ ]:
# Loading dataset
promoter_dataset = load_dataset("darlednik/geneb-tasks", "NT_promoter_all")
promoter_dataset

promoter_dataset["train"][0]

README.md:   0%|          | 0.00/24.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

tasks/InstaDeepAI_nucleotide_transformer(…): reconstructing file:   0%|          |  0.00B / 16.2MB            

tasks/InstaDeepAI_nucleotide_transformer(…): downloading bytes:           |  0.00B            

test.csv:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/53276 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5920 [00:00<?, ? examples/s]

{'text': 'ATAAAGCAACAGGAGGACACCGGCTCTCGCGTCATTAGGTCACACAATGTCTCAGTTTAAGGCGGGTGGGGTGGGCCAGACTCACAGCTGAGATAGAATTATTGCGAGATAACTCTGTTTAAGTAGCTAGGCTTGAAAATGGACGGTGAGTACCCTAGGACGTGCCCCAGGGCCAGTGGGATAAAGGTACACGCCGGCGGGGGCATGAGTCAGGGGTTTGCGAGCGAAAAAAGGAGCATTGGTTGCTACTGCGTTAGTGGGAACGTAGCTGGTCGCAGAGGGCACCTAATGCTGGCACCC',
 'label': 0}

## Tokenization

In [ ]:
# Function to generate overlapping k-mers from a given sequence
def seq2kmer(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq) - k + 1)])

# Function to tokenize a given sequence
def tokenize_promoter(example):
    # Convert a batch of raw strings to spaced k-mers
    kmer_texts = [seq2kmer(seq) for seq in example["text"]]

    return tokenizer(
        kmer_texts,
        truncation=True,
        padding="max_length",
        max_length=300
    )


tokenized_promoter = promoter_dataset.map(tokenize_promoter, batched=True)


Map:   0%|          | 0/53276 [00:00<?, ? examples/s]

Map:   0%|          | 0/5920 [00:00<?, ? examples/s]

In [ ]:
# Train/Test dataset with 80-20 split
train_promoter = tokenized_promoter["train"].shuffle(seed=42).select(range(8000))
valid_promoter = tokenized_promoter["test"].shuffle(seed=42).select(range(1000))

promoter_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  359MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: zhihan1996/DNA_bert_6
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## LoRA: Low-Rank Adaptation of the Trunk

We now wrap the freshly loaded `promoter_model` with `peft`. Three knobs define the adapter, three more are worth naming explicitly: the **task type**, the **target modules**, and the head policy:

- `r=16` : the rank of every low-rank update: each 768×768 target matrix gets a 768×16 `A` and a 16×768 `B`, ~25k trainable parameters instead of ~590k. Low **r** (4 to 24) are OK for simple adjustment, high r (24 to 256) for large adjustement.
- `lora_alpha=16` : the update actually applied is `(alpha / r) · B @ A` = `1 · B @ A`; tuning `alpha` effectively rescales the learning rate of the adapters without touching their size.
- `lora_dropout=0.05` : dropout on the adapter input path, a usual regularizer against overfitting the training sequences
- `target_modules=["query", "value"]` :  adapters wrap the `query` and `value` projections of every one of the 12 attention blocks.
    - This is the **default value** for LoRA. The idea is that attention is where token-to-token routing happens and changing it will change what the model *attends to*
    - alternative: `"all-linear"` (all modules) typical for QLoRA
- `task_type=SEQUENCE_CLASSIFICATION` : tells `peft` the model is a classifier: the classification head is kept **fully trainable** alongside the adapters


In [ ]:
# Wrap the pretrained model: every original weight stays frozen,
# trainable low-rank matrices are injected into the q and v projections
lora_config = LoraConfig(
    task_type=TaskType.SEQUENCE_CLASSIFICATION,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["query", "value"],
)

promoter_model = get_peft_model(promoter_model, lora_config)
promoter_model.print_trainable_parameters()

## Batch Size - GPU Memory Under LoRA

The nb03 analysis split peak memory into a **static cost** (~16 bytes per parameter in fp32: weights + gradients + the two AdamW states) and a **dynamic cost** (saved activations, linear in batch size, quadratic in sequence length). LoRA changes these two terms very differently:

- **Static cost collapses.** A frozen parameter never allocates gradients or AdamW states that saves the full ~12 bytes/parameter, not just a fraction. With ~99% of the trunk frozen, the bill drops from ~16 to roughly ~4 bytes per average parameter (the weights themselves) +  the 12 bytes for the ~1M trainable ones.
- **Dynamic cost barely changes.** Backpropagation still runs *through* every frozen layer to reach the adapters injected there: the saved activations, the term that grows with `batch × s²`, are almost identical to full fine-tuning. This is the classic LoRA caveat: it shrinks checkpoints and static memory, but does not divide VRAM by 100.

The same probe as nb03 applies unchanged. On your machine it now runs on the PEFT-wrapped model and measures the LoRA reality.

```
peak(b) = static + slope × b        →   safe_max_batch ≈ 0.8 × (GPU_total − static) / slope
```

Because the static part is lower, `safe_max_batch` comes out higher than in nb03. 

> Note: We deliberately keep batch 32 (identical to nb03) in the run below  so the two notebooks remain a single-variable comparison.

In [ ]:
# Empirical batch-size probe: peak memory (fwd + bwd + one AdamW step) at two batch sizes
def probe_batch_memory(model, batch_size, seq_len=300, vocab_size=4101):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    input_ids = torch.randint(5, vocab_size, (batch_size, seq_len), device="cuda")
    attention_mask = torch.ones_like(input_ids)
    labels = torch.randint(0, 2, (batch_size,), device="cuda")

    output = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    output.loss.backward()
    optimizer.step()
    peak_gb = torch.cuda.max_memory_allocated() / 1e9

    del optimizer, output, input_ids, attention_mask, labels
    torch.cuda.empty_cache()
    return peak_gb

if torch.cuda.is_available():
    b1, b2 = 4, 16
    m1 = probe_batch_memory(promoter_model.cuda(), b1)
    m2 = probe_batch_memory(promoter_model, b2)
    promoter_model.cpu()

    slope = (m2 - m1) / (b2 - b1)      # GB added by each extra sequence in the batch
    static = m1 - slope * b1           # weights + gradients + AdamW states + CUDA context
    gpu_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    max_batch = int(0.8 * (gpu_total - static) / slope)

    print(f"GPU total:   {gpu_total:.1f} GB")
    print(f"Static cost: {static:.2f} GB")
    print(f"Per sample:  {slope:.3f} GB (at 300 tokens per sequence)")
    print(f"Safe max batch (80% budget): {max_batch}")
else:
    print("No GPU detected — the probe needs CUDA.")
    print("Expect on a 16 GB GPU: static ~1-1.5 GB (no grads/optimizer states for frozen weights),")
    print("~0.1-0.2 GB per sample at 300 tokens, i.e. a safe batch above nb03's 32-64.")

## Evaluation Metrics

We use the same as before

In [ ]:
# Function to compute evaluation metrics
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    f1 = f1_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "f1": f1
    }


## TrainingArguments adapted to LoRA

We will focus on what's changed with respect to the full fine-tuning of notebook 03.

- `output_dir` : with LoRA each epoch checkpoint holds only the adapters + head (a few MB), not the ~360 MB model
- `learning_rate=2e-4` : the one argument that *must* change with LoRA. Full fine-tuning lives at 1e-5 to 5e-5 (nb03): a bigger step would overwrite pre-trained features (catastrophic forgetting). LoRA cannot overwrite the trunk at all so forgetting is off the table and the classic LoRA band is 1e-4 to 3e-4: roughly **10x the full-FT rate**
- `per_device_train_batch_size=32` : unchanged from nb03 for comparison purpose (the probe above says we could afford more)


In [ ]:
promoter_training_args = TrainingArguments(
    output_dir="./promoter_results_lora",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

promoter_trainer = Trainer(
    model=promoter_model,
    args=promoter_training_args,
    train_dataset=train_promoter,
    eval_dataset=test_promoter,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

## Fine-Tuning the Adapters

Observe:
- training loss — it should drop faster than nb03's 2e-5 run
- evaluation metrics — the target is nb03's full fine-tuning accuracy, reached with ~1% of the trainable weights
- checkpoint saving — MB-sized adapter checkpoints instead of hundreds of MB
- runtime behavior

In [ ]:
promoter_trainer.train()

promoter_results = promoter_trainer.evaluate()
promoter_results

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.341849,0.333707,0.855500,0.855393


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.341849,0.333707,1,0.855500,0.855393


{'eval_loss': 0.3337066173553467,
 'eval_accuracy': 0.8555,
 'eval_f1': 0.8553929276324432}

## Saving and Reloading the Adapter

Critical for:
- reproducibility
- downstream inference
- sharing checkpoints

With LoRA the *artifact* changes shape: `save_model` writes only the adapters and the head — a few MB that travel with the hub's trunk, instead of a ~360 MB full checkpoint. Reloading is a two-step sandwich: fresh pretrained trunk + adapter on top, then `merge_and_unload()` folds `W + (alpha/r)·B@A` back into the weights so inference is a plain `nn.Module` at full-fine-tuning speed.

In [ ]:
# Saving the adapter + head only - a few MB, the trunk stays on the hub
promoter_trainer.save_model("./promoter_lora_adapter")
tokenizer.save_pretrained("./promoter_lora_adapter")

# Loading back: fresh pretrained trunk, adapter on top
base_trunk = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
peft_promoter_model = PeftModel.from_pretrained(base_trunk, "./promoter_lora_adapter")

# Fold the low-rank updates into the trunk weights (W' = W + (alpha/r) * B @ A)
# -> inference runs as a plain model, at full fine-tuning speed
loaded_promoter_model = peft_promoter_model.merge_and_unload()

## Inference
We now perform prediction on unseen sequences.


In [ ]:
# Method to perform prediction on a sequence using the fine-tuned model
def promoter_predict(sequence):

    inputs = tokenizer(
        sequence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=300
    )

    with torch.no_grad():
        outputs = loaded_promoter_model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()
    return prediction


In [ ]:
# Run prediction
sample_sequence = promoter_dataset["test"][4000]["text"]

prediction = promoter_predict(sample_sequence)

print("Predicted label:", prediction)
print("Actual label:", promoter_dataset["test"][4000]["label"])


Predicted label: 1
Actual label: 1


## Probing the Representations: Before vs After LoRA

The classifier head tells us *that* LoRA worked — not *where inside the network* promoter information lives, or what the pre-trained model already knew.

**Probing** answers this: freeze the hidden states of each layer, train a tiny logistic regression on top of those frozen embeddings, and read its accuracy. A probe measures how *linearly accessible* the task information is at a given depth, without touching the model itself.

We compare two conditions on the *same* sequences:

- **Base**: fresh `zhihan1996/DNA_bert_6` from the hub — what the pre-trained features alone can support
- **LoRA-tuned**: the merged adapter model we just saved

And instead of a single lucky train/test split we run **5-fold stratified cross-validation** per layer, reporting mean ± std. (Further reading: [Hewitt & Liang, Designing and Interpreting Probes with Control Tasks](https://arxiv.org/abs/1905.05950).)

In [ ]:
# Extract pooled embeddings for every layer of a model, on the same sequences for both conditions
PROBE_SIZE = 1000

device = "cuda" if torch.cuda.is_available() else "cpu"

def get_layerwise_embeddings(model, sequences, batch_size=32):
    # Returns a list of (num_layers + 1) arrays of shape (n_sequences, hidden_size):
    # index 0 = input embedding layer, 1..12 = transformer layers
    model.eval().to(device)
    per_layer = None

    for start in range(0, len(sequences), batch_size):
        batch_texts = [seq2kmer(seq) for seq in sequences[start:start + batch_size]]
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=300,
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        mask = inputs["attention_mask"].unsqueeze(-1)  # (batch, seq_len, 1)
        if per_layer is None:
            per_layer = [[] for _ in outputs.hidden_states]

        # Attention-mask-aware mean pooling over real (non-padding) tokens
        for layer_idx, hidden in enumerate(outputs.hidden_states):
            summed = (hidden * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1)
            per_layer[layer_idx].append((summed / counts).cpu().numpy())

    model.cpu()
    if device == "cuda":
        torch.cuda.empty_cache()
    return [np.concatenate(chunks, axis=0) for chunks in per_layer]


probe_dataset = test_promoter.select(range(PROBE_SIZE))  # already shuffled when created
probe_sequences = probe_dataset["text"]
probe_labels = np.array(probe_dataset["label"])

base_probe_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

probe_embeddings = {
    "Base": get_layerwise_embeddings(base_probe_model, probe_sequences),
    "LoRA-tuned": get_layerwise_embeddings(loaded_promoter_model, probe_sequences),
}
del base_probe_model

n_layers = len(probe_embeddings["Base"])
print(f"Layers per model (incl. input embeddings): {n_layers}")
print(f"Embedding shape: {probe_embeddings['Base'][0].shape}")
print(f"Probed sequences: {len(probe_sequences)}")

In [ ]:
# One linear probe per layer per condition, scored with 5-fold stratified cross-validation
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

probe_results = {}
for condition, layers in probe_embeddings.items():
    means, stds = [], []
    for embeddings in layers:
        scores = cross_val_score(
            LogisticRegression(max_iter=1000),
            embeddings,
            probe_labels,
            cv=cv,
            scoring="accuracy",
        )
        means.append(scores.mean())
        stds.append(scores.std())
    probe_results[condition] = (np.array(means), np.array(stds))

print(f"{'Layer':>16} | {'Base':>13} | {'LoRA-tuned':>13}")
for i in range(n_layers):
    name = "Input embeddings" if i == 0 else f"Layer {i}"
    base_m, base_s = probe_results["Base"][0][i], probe_results["Base"][1][i]
    ft_m, ft_s = probe_results["LoRA-tuned"][0][i], probe_results["LoRA-tuned"][1][i]
    print(f"{name:>16} | {base_m:>7.3f} ±{base_s:.3f} | {ft_m:>7.3f} ±{ft_s:.3f}")

plt.figure(figsize=(8, 4))
for condition, color in zip(["Base", "LoRA-tuned"], ["tab:blue", "tab:orange"]):
    means, stds = probe_results[condition]
    plt.errorbar(range(n_layers), means, yerr=stds, fmt="o-", capsize=3,
                 color=color, label=condition)
plt.xlabel("Layer (0 = input embeddings, higher = deeper)")
plt.ylabel("Probe accuracy (mean ± std)")
plt.title("Where does promoter-relevant signal live - before vs after fine-tuning?")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### How to Read the Curves

- **The base curve is already well above chance (0.5)** — genome pre-training packed promoter-relevant information into the features, even though the model never saw a promoter label. This is the foundation LoRA builds on: it only needs to *redirect* accessible signal, not create it.
- **The LoRA lift is real but subtler than nb03's full fine-tuning.** Full FT reshaped every layer through every weight; here the only trainable path through the trunk runs through 24 rank-16 adapters on `query`/`value`. Expect the fine-tuned curve to sit above the base one — usually by slightly less, and with the gain concentrated in the layers where task-specific routing changes matter most.
- **A merged adapter changes hidden states only through attention.** The MLPs and the `key`/`output` projections are frozen weights; differences between the two curves are attention's doing alone — a sharper decomposition of "where the task lives" than full fine-tuning allowed.